<a href="https://www.kaggle.com/code/cartelsmith/step-by-step-building-a-neural-network?scriptVersionId=334000813" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Import Statements

In [ ]:
import pandas as pd
import seaborn as sns
sns.set_theme(style='white')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Loading Data
import kagglehub
from kagglehub import KaggleDatasetAdapter
file_path = "apple_products_pricing_2020_2026.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "ashyou09/apple-products-pricing-dataset-2020-2026",
  file_path)

# General Dataset Info

In [ ]:
print(df.info())

In [ ]:
display(df.head(10))

In [ ]:
drop_cols =['Launch_Price_INR','Current_Price_INR', 'Date']
df= df.drop(columns=drop_cols)
print(df.columns)

In [ ]:
# Dataframe Housekeeping

print('Unique values by column:')
display(df.nunique())    # Viewing unique values by col
to_cat_cols = [col for col in df.columns if df[col].nunique() < 5]
df[to_cat_cols] = df[to_cat_cols].astype('category') # Converting low-unique cols to 'category' dtype

print('\n\nConverted Dtypes df:')
display(df.dtypes)

In [ ]:
duplicates = int(df.duplicated().sum())     # Checking for Duplicate Rows
print(f"\n\nThere are {duplicates} duplicate rows in the dataframe.")

# Feature Selection

In [ ]:
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import TargetEncoder, RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn import set_config
set_config(transform_output="pandas")
from sklearn.compose import ColumnTransformer

X = df.copy()       # Features + Target
y = X.pop('Rating') # Removing Target col from X

print(f"\nThe shapes of 'X' and 'y', respectively, are: {X.shape}, {y.shape}.")

category_loc = df.columns.get_indexer(to_cat_cols) # Getting indicies of the categorical cols

# 🍭 I Eat Snacks (Impute, Encode, Scale) - Required for Mutual Information exploration
missing = X.isnull().sum()
print(f'\n\nMissing values per column:\n{missing}')

# 1. Imputing 'Sale_Event'
X['Sale_Event'] = X['Sale_Event'].cat.add_categories("Unknown")
X['Sale_Event'] = X['Sale_Event'].fillna('Unknown')

# 2. Encoding and Scaling X
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

col_transformer = ColumnTransformer(transformers=[('numeric', RobustScaler(), num_cols),
                                        ('categorical', TargetEncoder(random_state=12), cat_cols)],
                          remainder='passthrough')

X_ies = col_transformer.fit_transform(X, y)

# Post-transformation sanity check
print("\n\nX_ies statistics (Post Transformation):")
display(X_ies.describe().T) 
print('👍🏽 Everything looks good!')

In [ ]:
mutual_inf = mutual_info_regression(X_ies,y, discrete_features=category_loc, random_state=12)

mi_features = {'Features': X_ies.columns,
               'MI_Score': mutual_inf.round(3)
               }
mi_features = pd.DataFrame(mi_features).sort_values(by='MI_Score', ascending=False).reset_index(drop=True)

In [ ]:
display(mi_features)

# Charting Features by Mutual Info Score

sns.barplot(x=mi_features['MI_Score'],y=mi_features['Features'],
            orient='h', width =1, palette= 'rainbow')

plt.title("Mutual Info Scores", pad = 14, fontweight='bold', fontsize=12)
plt.show()

# Selecting only the features that have a MI Score > .001
modeling_feats = [feat for feat, score in zip(mi_features['Features'], mi_features['MI_Score']) if score > 0.01]

# Dropping prefix from column name
modeling_feats_clean = [col.split('__', 1)[-1] for col in modeling_feats]

# Feature Review

In [ ]:
# Viewing Selected Features 
sns.pairplot(X, corner=True)

mosaic = [['A', 'A', 'B'],
          ['D', 'C','C'],
          ['D', 'C','C']
         ]

fig, axes = plt.subplot_mosaic(mosaic=mosaic, figsize=(18,10))
sns.boxplot(x=X.Condition, y=y, ax=axes['A']);
sns.regplot(x=X.Current_Price_USD, y=y, ax=axes['C'])
sns.regplot(x=X.Discount_Pct, y=y, ax=axes['B'])
sns.regplot(x=X.Reviews_Count, y=y, ax=axes['D'])
plt.tight_layout()
plt.show()

sns.regplot(x=X.Launch_Price_USD, y=X.Current_Price_USD);

# Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

Xtrain,Xtest,ytrain,ytest = train_test_split(X_ies[modeling_feats], y, train_size=.75, random_state=12)

# ***Model Architecture***
  

In [ ]:
from keras.layers import Dense
from keras.models import Sequential
from keras.callbacks import EarlyStopping
from keras.utils import set_random_seed
set_random_seed(12)

model = Sequential([Dense(4,activation='relu',input_shape=(4,)),
                    Dense(2,activation='relu'),
                    Dense(1,activation='linear')])

model.compile(optimizer='adam',
              loss='mse',
              metrics=['mae'])

# View Model
print(model.summary())

# ***Model Evaluation***

In [ ]:
# 1. Define the Early Stopping rule
early_stopping = EarlyStopping(
    monitor='val_loss',          # metric to watch
    patience=7,                 # How many epochs to wait before stopping
    restore_best_weights=True    # Roll back to the best weights if stopped
)

# 2. Fit model and pass Early Stopping to callbacks
history = model.fit(x= Xtrain, y=ytrain,
                    batch_size=64, epochs=300,
                    validation_data=(Xtest, ytest),
                    callbacks=[early_stopping],        
                    )

In [ ]:
results = pd.DataFrame(history.history) # Results Dataframe

display(results.head())
print('\n\n')
print(results.info())

In [ ]:
train_loss = results['loss']
val_loss = results['val_loss']

# Charting Model Training 
sns.set_style('ticks')
sns.lineplot(x=results.index, y =train_loss, label ='Training Loss', ls ="dashed", lw=2.5, color="#963aecbd")
sns.lineplot(x=results.index, y =val_loss, label ='Validation Loss', lw =2.5, color = "#09ac5aff")

plt.title('Model Performance', pad=10, fontweight ='bold', fontsize= 15)
plt.xlabel('Epochs')
plt.ylabel('Mean Squared Error')
#plt.ylim(0,10)
plt.show()